# Environment Requirements
To run this repository successfully, the following environment is required:
- **GPU**: NVIDIA G4 with 95GB High RAM
- **Python**: 3.10 or higher
- **CUDA**: 12.1 or higher
- **PyTorch**: 2.1 or higher

In [ ]:
import os
import subprocess
import shutil

repo_url = "https://github.com/pandevim/scholera-coding-assessment-ai-engineer.git"
base_path = "/content"
repo_dir = "scholera-coding-assessment-ai-engineer"
repo_path = os.path.join(base_path, repo_dir)

os.chdir(base_path)

# Check if directory exists and is a git repo
is_repo = os.path.isdir(os.path.join(repo_path, ".git"))

try:
    if is_repo:
        print(f"Directory '{repo_dir}' exists. Attempting to update...")
        try:
            subprocess.run(["git", "-C", repo_path, "fetch", "--all"], check=True)
            subprocess.run(["git", "-C", repo_path, "reset", "--hard", "origin/main"], check=True)
            print("Successfully reset to origin/main.")
        except subprocess.CalledProcessError:
            # If reset fails (e.g. branch is named differently), try a standard pull
            subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    else:
        if os.path.exists(repo_path):
            shutil.rmtree(repo_path)
        print(f"Cloning {repo_url}...")
        subprocess.run(["git", "clone", repo_url], check=True)

    os.chdir(repo_path)
    print(f"Current working directory: {os.getcwd()}")

except subprocess.CalledProcessError as e:
    print(f"Git operation failed: {e}")

In [ ]:
%%capture
%pip install --upgrade --force-reinstall --no-deps "transformers[serving] @ https://github.com/huggingface/transformers/archive/refs/heads/main.zip"

In [ ]:
%%capture
%pip install -r requirements.txt

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
os.environ["HF_DATASET_REPO"] = "pandevim/cs584-tutor-index"

In [ ]:
os.environ["VLM_MODEL_ID"] = "Qwen/Qwen3.6-27B"
os.environ["SUMMARIZER_MODEL_ID"] = "Qwen/Qwen3.6-35B-A3B-FP8" # "Qwen/Qwen3.5-4B"
os.environ["EMBED_MODEL_ID"] = "Qwen/Qwen3-Embedding-8B"

In [ ]:
os.environ["RERANKER_MODEL_ID"] = "Qwen/Qwen3-Reranker-8B" # "Qwen/Qwen3-Reranker-4B"
os.environ["GENERATOR_MODEL_ID"] = "Qwen/Qwen3.5-9B" # "Qwen/Qwen3.6-35B-A3B-FP8"
os.environ["SMALL_LM_MODEL_ID"]  = "Qwen/Qwen3.5-4B"

In [ ]:
# Mount Drive once per session
from google.colab import drive
drive.mount('/content/drive')

# Point HF caches at Drive (set BEFORE any HF library usage)
import os
os.environ["HF_HOME"]              = "/content/drive/MyDrive/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/content/drive/MyDrive/hf_cache/hub"

Mounted at /content/drive


In [ ]:
!hf auth login

In [ ]:
import sys
from tutor import Tutor

t = Tutor()   # ~10 min — load all four models once

In [ ]:
%pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from pyngrok import ngrok

# Paste your token here
authtoken = "YOUR_NGROK_TOKEN_HERE"

!ngrok config add-authtoken {authtoken}
print("Authtoken configured successfully.")

In [ ]:
from pyngrok import ngrok
import nest_asyncio, uvicorn, threading
nest_asyncio.apply()
from server import app, _attach_existing_tutor
_attach_existing_tutor(t)
public_url = ngrok.connect(8000).public_url
print("public URL:", public_url)
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()